# Do `aoutools`' public functions work on the real Workbench?

Run this on the **All of Us Researcher Workbench**, on a Hail Genomic
Analysis environment.

`calculate_prs`, `calculate_prs_batch`, and `calculate_pgs` are the three
functions users actually call, and **no offline test can reach any of them**:
each one hard-raises unless `output_path` starts with `gs://`. So the GCS
writer, the local-file staging, and the PGS Catalog download are verified
nowhere but here.

This notebook is the companion to `validate_scoring_on_aou.ipynb`, which
checks that the *numbers* are right. This one checks that the **plumbing**
works: files download, parse, stage to the bucket, score, and come back as a
readable CSV.

It is deliberately cheap -- a small cohort and a small score.

## Setup

`init_hail()` supplies the requester-pays billing project (from
`$GOOGLE_PROJECT`) and sets the reference genome *after* `hl.init`, which is
now the only supported way.

`get_vds_path()` and `get_workspace_bucket()` exist because the current All of
Us platform (Verily) **no longer exports `WGS_VDS_PATH` or
`WORKSPACE_BUCKET`**. Code that reads them directly raises `KeyError` on a
fresh workspace.

`get_workspace_bucket()` recovers the bucket from the **gcsfuse mount table**
when the variable is missing, and warns with the name it found so a wrong guess
is visible rather than silent.

> **Exporting the variable from a Jupyter terminal will not work.** The
> terminal and the kernel are sibling children of the Jupyter server, which
> captured its environment when it started -- so the kernel never sees a
> variable you export later, and restarting the kernel re-inherits that same
> environment. That is why `!echo $WORKSPACE_BUCKET` comes back empty.
>
> To set it yourself, do it *inside* a notebook cell, which mutates the
> kernel's own environment:
>
> ```python
> import os
> os.environ["WORKSPACE_BUCKET"] = "gs://your-bucket"
> ```
>
> To make that survive kernel restarts, put those two lines in
> `~/.ipython/profile_default/startup/00-aou-env.py` -- every kernel runs that
> file at startup.


In [ ]:
# If aoutools isn't already installed in this environment:
# !pip install -q git+https://github.com/dokyoonkimlab/aoutools.git@dev

import pathlib

import hail as hl
import hailtop.fs as hfs
import pandas as pd

from aoutools import get_vds_path, get_workspace_bucket, init_hail
from aoutools.prs import (
    PRSConfig,
    calculate_pgs,
    calculate_prs,
    calculate_prs_batch,
    download_pgs,
    read_prs_weights,
)

init_hail()

VDS_PATH = get_vds_path()
BUCKET = get_workspace_bucket()
OUT = f"{BUCKET}/prs_validation"

print("VDS   :", VDS_PATH)
print("output:", OUT)

## A small cohort

`calculate_prs` reads only the loci its weights name, so the cost is driven by
the number of samples far more than the size of the VDS. A few hundred is
plenty to prove the plumbing.

In [ ]:
N_SAMPLES = 200

vds = hl.vds.read_vds(VDS_PATH)

sample_ids = vds.variant_data.cols().head(N_SAMPLES).s.collect()
samples_ht = hl.Table.parallelize(
    [{"s": s} for s in sample_ids], hl.tstruct(s=hl.tstr)
).key_by("s")
vds = hl.vds.filter_samples(vds, samples_ht)

print(f"{len(sample_ids)} samples")

## 1. `download_pgs`

Note the signature: `download_pgs` is **keyword-only** and returns `None` --
it writes files and tells you nothing. Pass `outdir` and `pgs`.

Two things will bite you here, and both did:

* The output directory **must already exist**. Neither `aoutools` nor the
  underlying CLI creates it. Hence the explicit `mkdir`.
* **`overwrite_existing_file` defaults to `False`**, which makes an already
  present file a hard *error* -- and since the downloads run concurrently, it
  aborts the **whole batch**, not just that one score. So a second run of this
  cell fails unless you pass `True`. We do.

### Why these two scores

Both were **checked**, not guessed. Each is small, fully harmonized to GRCh38,
and free of duplicate variants:

| | variants | unmapped | duplicates |
|---|---|---|---|
| PGS000746 | 1,940 | 0 | 0 |
| PGS000750 | 43 | 0 | 0 |

The earlier draft of this notebook paired PGS000746 with **PGS000747**, which
is 375,822 variants -- too big for a cheap check -- and contains 20 variants
that failed harmonization. Those 20 have a null chromosome *and* a null
position, so they all collapse onto the same `variant_id` (`null_null_A_G`) and
`read_prs_weights` rejected the entire file as duplicated. That was a real bug
and it is now fixed: unmapped variants are dropped with a warning. Almost every
large PGS score has a few of them.


In [ ]:
PGS_IDS = ["PGS000746", "PGS000750"]

# The directory must exist first: the pgscatalog CLI checks, and refuses to
# create it.
pgs_dir = pathlib.Path.home() / "pgs_downloads"
pgs_dir.mkdir(parents=True, exist_ok=True)
print("downloading to:", pgs_dir)

download_pgs(
    outdir=str(pgs_dir),
    pgs=PGS_IDS,
    build="GRCh38",
    # Without this, re-running the cell raises FileExistsError and takes the
    # whole batch down with it.
    overwrite_existing_file=True,
)

files = sorted(pgs_dir.rglob("*.txt.gz"))
for f in files:
    print(f.name, f"{f.stat().st_size / 1024:.0f} KB")

assert len(files) == len(PGS_IDS), (
    f"expected {len(PGS_IDS)} scoring files, got {len(files)}"
)

## 2. `read_prs_weights`

The real parser, against real hail. `tests/prs/` mocks hail entirely, so the
header dispatch, the `column_map`, `_validate_alleles`, and
`_check_duplicated_ids` have never actually run outside this notebook.

The `column_map` below is the one `calculate_pgs` uses internally for PGS
Catalog files. `hm_chr` / `hm_pos` are the **harmonized** coordinates; note
that harmonization remaps positions but does **not** flip the effect allele
onto the ALT -- which is why orientation has to be resolved per row against
the VDS.

A local path is staged to `$WORKSPACE_BUCKET/data/...` automatically, because
Hail's Spark cluster cannot read the notebook's local filesystem. That
staging is `_stage_local_file_to_gcs`, and this is the only place it runs.

In [ ]:
PGS_COLUMN_MAP = {
    "chr": "hm_chr",
    "pos": "hm_pos",
    "effect_allele": "effect_allele",
    "noneffect_allele": "other_allele",
    "weight": "effect_weight",
}

weights_map = {}
for f in files:
    score_name = f.name.split(".")[0]
    weights_map[score_name] = read_prs_weights(
        file_path=str(f),  # local path -> staged to GCS for us
        header=True,
        column_map=PGS_COLUMN_MAP,
        delimiter="\t",
        comment="#",
        validate_alleles=True,
        missing="",
        force=True,
    )

for name, ht in weights_map.items():
    print(f"{name}: {ht.count():,} variants")
    ht.show(3)

### Did the effect allele actually come mixed?

The whole per-row orientation design rests on the claim that PGS Catalog files
do **not** harmonize the effect allele onto the ALT. Check it: count how many of
this score's rows name a REF base as the effect allele, once joined against the
VDS.

If the answer is 0, the claim is wrong and the `2w` hom-ref offset is dead code
that never runs. If it is nonzero, orientation genuinely is a per-row property
-- and the old file-level `ref_is_effect_allele` flag was silently dropping
every row that disagreed with it.

> **This cell is a diagnostic, not part of the library.** It reads the VDS's own
> REF at each locus, which scoring does not need to do separately -- the
> calculator gets it for free from the row it is already joined to. An earlier
> draft used `group_by(locus)` here, which **shuffles the VDS across
> partitions**; that is the one thing `aoutools` is built never to do, and it
> was very slow. It is now a key-prefix join, like the one the calculator uses.
>
> A sample of loci is enough to answer "is this file mixed?", so it only reads
> `N_PROBE` of them rather than all of the score's variants.


In [ ]:
# `read_prs_weights` returns chr/pos COLUMNS, not a locus -- the locus is built
# later, by the scoring path. Use the library's own preparation step so this
# check sees exactly the table the calculator would join.
from aoutools.prs._calculator_utils import _validate_and_prepare_weights_table

N_PROBE = 300  # loci to sample; enough to tell a mixed file from a uniform one

score_name = list(weights_map)[0]
ht = _validate_and_prepare_weights_table(weights_map[score_name], PRSConfig())
ht = ht.head(N_PROBE).persist()
n_probed = ht.count()
print(f"{score_name}: probing {n_probed:,} loci")

# Read only those loci. This is what the calculator does per chunk.
intervals = ht.select(
    interval=hl.interval(ht.locus, ht.locus, includes_end=True)
).key_by("interval")
rows = hl.vds.filter_intervals(vds, intervals, keep=True).variant_data.rows()

# `rows` is keyed by (locus, alleles) and `ht` by locus, so `ht[rows.locus]` is
# a KEY-PREFIX join: no shuffle, one pass. The previous group_by(locus) here
# shuffled the whole VDS and was the reason this cell crawled.
matched = rows.annotate(w=ht[rows.locus])
matched = matched.filter(hl.is_defined(matched.w))

# Three buckets, not two. An effect allele that is not the REF is NOT thereby
# the ALT -- it may be an allele that does not exist at this locus at all, in
# which case the variant is a genuine mismatch and will not score. Counting
# `!= REF` as "ALT" hides that entirely, and this cell used to do exactly that.
alts = matched.alleles[1:]
counts = matched.aggregate(
    hl.struct(
        n=hl.agg.count(),
        n_ref_effect=hl.agg.count_where(
            matched.w.effect_allele == matched.alleles[0]
        ),
        n_alt_effect=hl.agg.count_where(alts.contains(matched.w.effect_allele)),
    )
)
n, n_ref, n_alt = counts.n, counts.n_ref_effect, counts.n_alt_effect
n_neither = n - n_ref - n_alt

print(f"\n{n:,} of the {n_probed:,} probed loci are present in the VDS")
print(f"  effect allele == REF     : {n_ref:,}")
print(f"  effect allele == an ALT  : {n_alt:,}")
print(f"  effect allele == NEITHER : {n_neither:,}  (cannot score)")

if n == 0:
    print("\nNo overlap -- try a larger N_PROBE or a different score.")
elif n_ref == 0:
    print(
        "\nNOTE: every effect allele in this sample is the ALT, so this score "
        "does not exercise the hom-ref offset. Try another PGS ID."
    )
else:
    print(
        f"\nConfirmed: {n_ref:,}/{n:,} rows name the REF as the effect allele. "
        "Orientation is a per-row property, exactly as assumed -- a file-level "
        "flag would have silently dropped one of these two groups."
    )

# The join the calculator actually performs is stricter than locus presence: it
# needs the weights' allele PAIR to equal the VDS row's (REF, ALT). So this
# probe is an upper bound on the match rate, and the gap between the two is
# worth a look. It is what exposed the sorted-vs-unsorted key bug: ~100% of
# loci were present here while `n_matched` below reported only ~48% of the
# score's variants, because a weights key sorted as [A, G] never met a VDS row
# keyed [G, A]. If those two numbers diverge again, suspect the join first.
print(
    f"\nUpper bound on the match rate from this sample: "
    f"{(n - n_neither) / n_probed:.1%}. Compare with `n_matched` below."
)

## 3. `calculate_prs` -- the `gs://` path no test can reach

In [ ]:
score_name = list(weights_map)[0]
single_out = f"{OUT}/single.csv"

result_path = calculate_prs(
    weights_table=weights_map[score_name],
    vds=vds,
    output_path=single_out,
    config=PRSConfig(include_n_matched=True),
)
print("wrote:", result_path)

with hfs.open(single_out, "r") as fh:
    single_df = pd.read_csv(fh)

print(single_df.head())
print(f"\n{len(single_df)} rows, {single_df.prs.notna().sum()} finite scores")
print(single_df.prs.describe())

assert len(single_df) == len(sample_ids), (
    f"expected one row per sample ({len(sample_ids)}), got {len(single_df)}"
)
assert single_df.prs.notna().all(), "a sample got no score at all"
assert single_df.prs.nunique() > 1, (
    "every sample got the SAME score -- that is not a PRS, it is a constant"
)
print("\nOK: one finite, varying score per sample.")

## 4. `calculate_prs_batch`

Batch masks with `hl.if_else` where the single-score path filters rows -- a
different mechanism that must produce the same number. Scoring the same score
both ways is the cheapest possible check on that.

In [ ]:
batch_out = f"{OUT}/batch.csv"

calculate_prs_batch(
    weights_tables_map=weights_map,
    vds=vds,
    output_path=batch_out,
    config=PRSConfig(include_n_matched=True),
)

with hfs.open(batch_out, "r") as fh:
    batch_df = pd.read_csv(fh)

print(batch_df.head())

merged = single_df[["person_id", "prs"]].merge(
    batch_df[["person_id", score_name]], on="person_id"
)
delta = (merged["prs"] - merged[score_name]).abs().max()
print(f"\nlargest single-vs-batch difference: {delta:.3g}")

assert set(weights_map) <= set(batch_df.columns), (
    f"expected one column per score, got {list(batch_df.columns)}"
)
assert delta < 1e-9, (
    "batch and single disagree. They use different mechanisms (if_else "
    "masking vs filter_rows) and must still produce identical scores."
)
print("OK: batch matches single exactly.")

## 5. `calculate_pgs` -- download, read, and score in one call

In [ ]:
pgs_out = f"{OUT}/pgs.csv"

calculate_pgs(
    vds=vds,
    output_path=pgs_out,
    pgs=PGS_IDS,
    build="GRCh38",
    config=PRSConfig(include_n_matched=True),
)

with hfs.open(pgs_out, "r") as fh:
    pgs_df = pd.read_csv(fh)

print(pgs_df.head())

merged = batch_df[["person_id", score_name]].merge(
    pgs_df[["person_id", score_name]], on="person_id", suffixes=("_b", "_p")
)
delta = (merged[f"{score_name}_b"] - merged[f"{score_name}_p"]).abs().max()
print(f"\nlargest calculate_pgs-vs-batch difference: {delta:.3g}")

assert delta < 1e-9, "calculate_pgs disagrees with the batch call it wraps"
print("OK: the end-to-end workflow reproduces the batch scores.")

## 6. The removed knobs are a hard error

Three `PRSConfig` parameters were deleted because each one silently lost data:

* `split_multi=False` selected a scoring path that zeroed every hom-ref sample
  at a REF-effect variant, reordering the cohort.
* `ref_is_effect_allele` declared orientation for a whole file when it is a
  per-row property, dropping every row that disagreed.
* `strict_allele_match=False` scored a weights row against a variant with
  different alleles.

They now raise `TypeError` rather than being quietly ignored. This is the
machine where someone's old notebook will actually hit that, so check it here.

In [ ]:
for knob in ["split_multi", "ref_is_effect_allele", "strict_allele_match"]:
    try:
        PRSConfig(**{knob: True})
    except TypeError as e:
        print(f"OK    {knob:22s} -> TypeError: {e}")
    else:
        raise AssertionError(
            f"{knob} was accepted. It was removed because it silently lost "
            "data; accepting it again means old notebooks keep producing "
            "wrong scores with no warning."
        )

## Done

Everything the offline suites cannot reach has now run against the real
Workbench: the PGS Catalog download, the real file parser, local-to-GCS
staging, all three public entry points, and the CSV writer.

For whether the *numbers* are right -- allele orientation, the hom-ref offset,
multi-allelic downcoding -- see `validate_scoring_on_aou.ipynb`.